## Teoría base (cómo leer los bloques de código)

Un agente decide **cuándo pensar** y **cuándo usar una herramienta**.

Bucle:
$$
\text{Objetivo} \rightarrow \text{Acción/tool} \rightarrow \text{Observación} \rightarrow \text{Respuesta}
$$

En los bloques clave, verifica:
- definición de tools,
- seguridad de llamadas (por ejemplo SQL de solo lectura),
- integración del resultado en la respuesta final.

## Guía de lectura (teoría ↔ práctica)

Un agente combina razonamiento + uso de herramientas (tools).

Esquema conceptual:
$$
\text{Pregunta} \rightarrow \text{Plan} \rightarrow \text{Tool Call} \rightarrow \text{Observación} \rightarrow \text{Respuesta}
$$

- En SQL, separar consulta de solo lectura (`SELECT`) mejora seguridad.
- **Qué observar**: cuándo el agente decide usar una tool y cómo integra el resultado.

# Herramientas de la calculadora

Como ejemplo vamos a conectar un agente con una *calculadora*. Para ellos vamos a crear una tool por cada operación matemática simple.

In [1]:
from smolagents import tool

# declaramos las herramientas que va a usar el agente

@tool
def add(a: float, b: float) -> float:
    """
    Adds two numbers together.
    
    Args:
        a (float): The first number.
        b (float): The second number.
    """
    return a + b

@tool
def subtract(a: float, b: float) -> float:
    """
    Subtracts the second number from the first.
    
    Args:
        a (float): The first number.
        b (float): The second number.
    """
    return a - b

@tool
def multiply(a: float, b: float) -> float:
    """
    Multiplies two numbers together.
    
    Args:
        a (float): The first number.
        b (float): The second number.
    """
    return a * b

@tool
def divide(a: float, b: float) -> float | str:
    """
    Divides the first number by the second. Returns an error message if division by zero is attempted.
    
    Args:
        a (float): The first number.
        b (float): The second number.
    """
    if b == 0:
        return "Error: Division by zero is not allowed."
    return a / b

/home/iker/miniconda3/envs/agents/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Ollama local

Vamos a hacer un ejemplo de calculadora usando smolagents y Ollama.

In [2]:
from smolagents import LiteLLMModel

# instanciamos el LLM que vamos a utilizar
# en nuestro caso uso de Ollama
model = LiteLLMModel(
    model_id="ollama-chat/qwen2.5-coder:7b",
    api_base="http://localhost:11434",
    temperature=0.2,
)

In [3]:
from smolagents import ToolCallingAgent

# instanciamos el agente con las herramientas declaradas
agent = ToolCallingAgent(
    model=model,
    tools=[add, subtract, multiply, divide]
)

In [4]:
# ejemplo de uso
agent.run("What is 15 multiplied by 3, then subtract 5 and finally divide by 2?")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is 15 multiplied by 3, then subtract 5 and finally divide by 2?                                            │
│                                                                                                                 │
╰─ LiteLLMModel - ollama-chat/qwen2.5-coder:7b ───────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



Error while generating output:
litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed 
model=ollama-chat/qwen2.5-coder:7b
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` 
Learn more: https://docs.litellm.ai/docs/providers

[Step 1: Duration 0.04 seconds]

AgentGenerationError: Error while generating output:
litellm.BadRequestError: LLM Provider NOT provided. Pass in the LLM provider you are trying to call. You passed model=ollama-chat/qwen2.5-coder:7b
 Pass model as E.g. For 'Huggingface' inference endpoints pass in `completion(model='huggingface/starcoder',..)` Learn more: https://docs.litellm.ai/docs/providers

In [ ]:
# para inspeccionar qué ha pasado internamente
agent.replay(detailed=False)

## Ollama cloud

In [ ]:
from smolagents import LiteLLMModel

model = LiteLLMModel(
    model_id="ollama_chat/gpt-oss:120b",
    api_base="https://ollama.com",
    temperature=0.2,
    api_key="<INSERT_YOUR_API_KEY_HERE>"
)

In [6]:
from smolagents import ToolCallingAgent

# instanciamos el agente con las herramientas declaradas
agent = ToolCallingAgent(
    model=model,
    tools=[add, subtract, multiply, divide]
)

In [7]:
agent.run("What is 15234234 multiplied by 43423, then subtract 2342345 and finally divide by 23123? Use the tools to calculate the answer.")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is 15234234 multiplied by 43423, then subtract 2342345 and finally divide by 23123? Use the tools to       │
│ calculate the answer.                                                                                           │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/gpt-oss:120b ───────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'multiply' with arguments: {'a': 15234234, 'b': 43423}                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: 661516142982

[Step 1: Duration 1.72 seconds| Input tokens: 1,374 | Output tokens: 89]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 28608476.43631882}                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: 28608476.43631882

Final answer: 28608476.43631882

[Step 3: Duration 1.97 seconds| Input tokens: 4,326 | Output tokens: 435]

28608476.43631882

In [8]:
from smolagents import GradioUI

GradioUI(agent).launch()

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://d133168489aebd255b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://d133168489aebd255b.gradio.live


## Gemini API

In [ ]:
from smolagents import OpenAIModel

model = OpenAIModel(
    model_id="gemini-2.5-flash",
    # Google Gemini OpenAI-compatible API base URL
    api_base="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key="<INSERT_YOUR_API_KEY_HERE>",
)

In [ ]:
from smolagents import ToolCallingAgent

# instanciamos el agente con las herramientas declaradas
agent = ToolCallingAgent(
    model=model,
    tools=[add, subtract, multiply, divide]
)

In [ ]:
agent.run("What is 15234234 multiplied by 43423, then subtract 2342345 and finally divide by 23123? Use the tools to calculate the answer.")

# Open AI API

In [ ]:
from smolagents import OpenAIModel
import os

os.environ["OPENAI_API_KEY"] = "<token>"

model = OpenAIModel(
    model_id="gpt-4o",
    api_base="https://api.openai.com/v1",
    api_key=os.environ["OPENAI_API_KEY"],
)

In [ ]:
from smolagents import ToolCallingAgent

# instanciamos el agente con las herramientas declaradas
agent = ToolCallingAgent(
    model=model,
    tools=[add, subtract, multiply, divide]
)

In [ ]:
agent.run("What is 15234234 multiplied by 43423, then subtract 2342345 and finally divide by 23123? Use the tools to calculate the answer.")

# Ejercicio

Haz un conjunto de herramientas de manera que el LLM interactúe con una base de datos SQLite. Las herramientas deben ser:
- `query_database`. Para ejecutar una SQL `SELECT` y devolver los (100 primeros) resultados.
- `show_tables`. Para listar las tablas de la base de datos.
- `describe_table`. Para describir la estructura de una tabla dada (sus columnas).

Se usa la base de datos de ejemplo Chinook_Sqlite.sqlite. 
- Info sobre la base de datos y modelo de datos: https://github.com/lerocha/chinook-database
- Ejemplos de consultas: https://github.com/LucasMcL/15-sql_queries_02-chinook/blob/master/chinook-queries.sql 

El último link usarlo para obtener consultas en lenguaje natural para probar vuestro agente.

In [10]:
import sqlite3
from pathlib import Path
from typing import Dict, Any

SQLITE_DB_PATH = Path("Chinook_Sqlite.sqlite")
if not SQLITE_DB_PATH.exists():
    SQLITE_DB_PATH = Path("dia4-2") / "Chinook_Sqlite.sqlite"

def get_connection():
    """Return connection to the sqlite database."""
    conn = sqlite3.connect(str(SQLITE_DB_PATH))
    conn.row_factory = sqlite3.Row
    return conn

@tool
def query_database(query: str) -> Dict[str, Any]:
    """Run a SELECT SQL query against the database and return results.

    Args:
        query: SELECT SQL query to execute.

    Returns:
        A dictionary with the query results (rows, columns, etc.).
    """
    sql = query.strip().rstrip(";")
    if not sql.lower().startswith("select"):
        return {"ok": False, "error": "Only SELECT queries are allowed."}

    try:
        with get_connection() as conn:
            cursor = conn.execute(sql)
            rows = cursor.fetchmany(100)
            columns = [description[0] for description in cursor.description] if cursor.description else []

        result_rows = [dict(row) for row in rows]
        return {
            "ok": True,
            "columns": columns,
            "rows": result_rows,
            "row_count": len(result_rows),
            "truncated": len(result_rows) == 100,
        }
    except Exception as error:
        return {"ok": False, "error": str(error)}

@tool
def show_tables() -> Dict[str, Any]:
    """Returns the list of tables in the database."""
    sql = "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"
    result = query_database(sql)
    if not result.get("ok"):
        return result

    table_names = [row["name"] for row in result["rows"]]
    return {"ok": True, "tables": table_names, "table_count": len(table_names)}

@tool
def describe_table(table_name: str) -> Dict[str, Any]:
    """Returns the schema of the specified table.
    Args:
        table_name: Name of the table to describe.
    Returns:
        A dictionary with the table schema information.
    """
    cleaned_name = table_name.strip().replace("'", "''")
    sql = f"PRAGMA table_info('{cleaned_name}')"

    try:
        with get_connection() as conn:
            cursor = conn.execute(sql)
            rows = cursor.fetchall()

        columns = [dict(row) for row in rows]
        if not columns:
            return {"ok": False, "error": f"Table '{table_name}' not found."}

        return {"ok": True, "table": table_name, "columns": columns}
    except Exception as error:
        return {"ok": False, "error": str(error)}

In [ ]:
from smolagents import LiteLLMModel, ToolCallingAgent

model = LiteLLMModel(
    model_id="ollama_chat/gpt-oss:120b",
    api_base="https://ollama.com",
    temperature=0.2,
    api_key="<INSERT_YOUR_API_KEY_HERE>"
)

agent = ToolCallingAgent(
    model=model,
    tools=[query_database, show_tables, describe_table],
)

agent.run("Who is the customer that spent the most money? Give me the name")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Who is the customer that spent the most money? Give me the name                                                 │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/gpt-oss:120b ───────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'show_tables' with arguments: {}                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: {'ok': True, 'tables': |'Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine',
'MediaType', 'Playlist', 'PlaylistTrack', 'Track'], 'table_count': 11}

[Step 1: Duration 8.55 seconds| Input tokens: 1,144 | Output tokens: 49]

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'Helena Holý'}                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Helena Holý

Final answer: Helena Holý

[Step 3: Duration 0.88 seconds| Input tokens: 3,802 | Output tokens: 246]

'Helena Holý'

In [12]:
from smolagents import GradioUI

# si queréis jugar con el agente en una interfaz gráfica
GradioUI(agent).launch()

* Running on local URL:  http://127.0.0.1:7860


KeyboardInterrupt: 